In [1]:
"""
Cluster validation, run after 03_segmentation.py.

Covers:
  1. Bootstrap stability of the k=3 solution (Adjusted Rand Index across
     resamples)
  2. Algorithm-independent cross-check (k-means vs. agglomerative
     clustering agreement)
  3. Explicit k=2 vs k=3 comparison to justify the k selection beyond
     the silhouette score alone (which was only marginally higher for
     k=3 -- see script 03 output)

Run: python 06_cluster_validation.py
Requires: pandas, numpy, scikit-learn, matplotlib
Input: findex_pakistan_with_clusters.csv (output of script 03)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score

RANDOM_STATE = 42
CSV_PATH = "data\\findex_pakistan_with_clusters.csv"
N_BOOTSTRAP = 100

data = pd.read_csv(CSV_PATH)
print(f"Loaded {data.shape[0]} rows, {data.shape[1]} columns")

drop_cols = ["adoption_tier", "adoption_tier_label", "cluster",
             "account", "dig_account", "anydigpayment", "saved", "fin22a",
             "account_fin", "account_mob"]
X = data.drop(columns=drop_cols)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

original_labels = data["cluster"].values

Loaded 1000 rows, 25 columns


In [2]:
# =======================================================================
# SECTION 1: BOOTSTRAP STABILITY OF THE k=3 SOLUTION
# =======================================================================
print("\n" + "=" * 70)
print(f"1. BOOTSTRAP STABILITY (k=3, {N_BOOTSTRAP} resamples)")
print("=" * 70)
print("Logic: refit k-means on a bootstrap resample, then use the refit")
print("model to LABEL THE FULL ORIGINAL SAMPLE. Compare that labeling to")
print("the original clustering via Adjusted Rand Index (ARI). ARI close")
print("to 1.0 = stable structure; close to 0 = structure is an artifact")
print("of the particular sample drawn.\n")

n = X_scaled.shape[0]
rng = np.random.RandomState(RANDOM_STATE)
ari_scores = []

for i in range(N_BOOTSTRAP):
    boot_idx = rng.choice(n, size=n, replace=True)
    X_boot = X_scaled[boot_idx]
    km_boot = KMeans(n_clusters=3, random_state=RANDOM_STATE + i, n_init=10)
    km_boot.fit(X_boot)
    # Label the FULL original sample with the bootstrap-fitted model
    boot_labels_on_full = km_boot.predict(X_scaled)
    ari = adjusted_rand_score(original_labels, boot_labels_on_full)
    ari_scores.append(ari)

ari_scores = np.array(ari_scores)
print(f"Bootstrap ARI: mean={ari_scores.mean():.3f}, std={ari_scores.std():.3f}")
print(f"95% percentile interval: [{np.percentile(ari_scores, 2.5):.3f}, "
      f"{np.percentile(ari_scores, 97.5):.3f}]")
print("(Note: k-means cluster ID labels are arbitrary/interchangeable across "
      "refits -- ARI correctly accounts for this via pairwise agreement, so no "
      "manual label-matching is needed.)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(ari_scores, bins=20, color="#2a9d8f", edgecolor="white")
ax.axvline(ari_scores.mean(), color="#e76f51", linestyle="--",
           label=f"Mean = {ari_scores.mean():.3f}")
ax.set_xlabel("Adjusted Rand Index (vs. original clustering)")
ax.set_ylabel("Frequency")
ax.set_title(f"Bootstrap Cluster Stability (k=3, {N_BOOTSTRAP} resamples)")
ax.legend()
plt.tight_layout()
plt.savefig("cluster_stability_bootstrap.png", dpi=150, bbox_inches="tight")
print("Saved: cluster_stability_bootstrap.png")
plt.close()

pd.DataFrame({"bootstrap_ari": ari_scores}).to_csv("cluster_stability_bootstrap.csv", index=False)



1. BOOTSTRAP STABILITY (k=3, 100 resamples)
Logic: refit k-means on a bootstrap resample, then use the refit
model to LABEL THE FULL ORIGINAL SAMPLE. Compare that labeling to
the original clustering via Adjusted Rand Index (ARI). ARI close
to 1.0 = stable structure; close to 0 = structure is an artifact
of the particular sample drawn.

Bootstrap ARI: mean=0.904, std=0.041
95% percentile interval: [0.847, 0.982]
(Note: k-means cluster ID labels are arbitrary/interchangeable across refits -- ARI correctly accounts for this via pairwise agreement, so no manual label-matching is needed.)
Saved: cluster_stability_bootstrap.png


In [3]:
# =======================================================================
# SECTION 2: ALGORITHM-INDEPENDENT CROSS-CHECK
# =======================================================================
print("\n" + "=" * 70)
print("2. ALGORITHM-INDEPENDENT CROSS-CHECK (k-means vs. agglomerative)")
print("=" * 70)

agg = AgglomerativeClustering(n_clusters=3, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)
ari_algo = adjusted_rand_score(original_labels, agg_labels)
print(f"ARI between k-means and Ward agglomerative clustering (both k=3): {ari_algo:.3f}")
print("(High agreement here means the 3-cluster structure is not an artifact")
print(" of k-means specifically, but recoverable by a different algorithm")
print(" with different assumptions about cluster shape.)")

pd.DataFrame({"metric": ["kmeans_vs_agglomerative_ari"], "value": [ari_algo]}) \
    .to_csv("cluster_algorithm_crosscheck.csv", index=False)


2. ALGORITHM-INDEPENDENT CROSS-CHECK (k-means vs. agglomerative)
ARI between k-means and Ward agglomerative clustering (both k=3): 0.740
(High agreement here means the 3-cluster structure is not an artifact
 of k-means specifically, but recoverable by a different algorithm
 with different assumptions about cluster shape.)


In [4]:
# =======================================================================
# SECTION 3: k=2 vs k=3 -- INTERPRETABILITY COMPARISON
#    (Silhouette score in script 03 was only marginally higher for k=3
#    than k=2: 0.1675 vs 0.1674. This section provides the substantive,
#    not just statistical, justification for reporting k=3.)
# =======================================================================
print("\n" + "=" * 70)
print("3. k=2 vs k=3 -- INTERPRETABILITY COMPARISON")
print("=" * 70)

km2 = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10)
labels_k2 = km2.fit_predict(X_scaled)
sil_k2 = silhouette_score(X_scaled, labels_k2)
sil_k3 = silhouette_score(X_scaled, original_labels)
print(f"Silhouette score: k=2 -> {sil_k2:.4f}  |  k=3 -> {sil_k3:.4f}")

data_temp = data.copy()
data_temp["cluster_k2"] = labels_k2

profile_k2 = data_temp.groupby("cluster_k2").agg(
    n=("cluster_k2", "size"),
    mean_adoption_tier=("adoption_tier", "mean"),
    pct_female=("female", "mean"),
    pct_rural=("is_rural", "mean"),
    pct_mobile_phone=("con1", "mean"),
    pct_internet_use=("internet_use", "mean"),
).round(3)
profile_k2["pct_female"] *= 100
profile_k2["pct_rural"] *= 100
profile_k2["pct_mobile_phone"] *= 100
profile_k2["pct_internet_use"] *= 100
profile_k2["pct_of_sample"] = (profile_k2["n"] / len(data_temp) * 100).round(1)

print("\nk=2 cluster profile:")
print(profile_k2.to_string())
profile_k2.to_csv("cluster_profile_k2_comparison.csv")

# Does k=2 collapse the k=3 "Digitally Excluded Women" and "Connected but
# Unengaged" clusters (both low-adoption but access-differentiated) into
# a single low-adoption group, losing the access-usage-gap distinction?
crosstab_k2_k3 = pd.crosstab(labels_k2, original_labels)
crosstab_k2_k3.index.name = "k=2 cluster"
crosstab_k2_k3.columns.name = "k=3 cluster"
print("\nCrosstab: k=2 clusters vs k=3 clusters (row=k2, col=k3):")
print(crosstab_k2_k3.to_string())
crosstab_k2_k3.to_csv("cluster_k2_vs_k3_crosstab.csv")

ari_k2_k3 = adjusted_rand_score(labels_k2, original_labels)
print(f"\nARI between k=2 and k=3 solutions: {ari_k2_k3:.3f}")
print("Interpretation to report: if the k=3 'Digitally Excluded Women' and")
print("'Connected but Unengaged' clusters map predominantly into a single")
print("k=2 cluster, this shows k=2 conflates two access-differentiated but")
print("both low-adoption groups -- precisely the distinction motivating H1.")
print("This substantive loss of the access-usage-gap distinction, not just")
print("the marginal silhouette difference, is the primary justification")
print("for selecting k=3 despite its narrow quantitative margin over k=2.")

print("\n" + "=" * 70)
print("Cluster validation complete. Files saved:")
print("  - cluster_stability_bootstrap.png / .csv")
print("  - cluster_algorithm_crosscheck.csv")
print("  - cluster_profile_k2_comparison.csv")
print("  - cluster_k2_vs_k3_crosstab.csv")
print("=" * 70)


3. k=2 vs k=3 -- INTERPRETABILITY COMPARISON
Silhouette score: k=2 -> 0.1675  |  k=3 -> 0.1675

k=2 cluster profile:
              n  mean_adoption_tier  pct_female  pct_rural  pct_mobile_phone  pct_internet_use  pct_of_sample
cluster_k2                                                                                                   
0           516               1.355        12.8       40.9              98.3              73.6           51.6
1           484               0.231        89.7       51.4              35.7              11.6           48.4

Crosstab: k=2 clusters vs k=3 clusters (row=k2, col=k3):
k=3 cluster    0    1    2
k=2 cluster               
0              1  389  126
1            430   54    0

ARI between k=2 and k=3 solutions: 0.623
Interpretation to report: if the k=3 'Digitally Excluded Women' and
'Connected but Unengaged' clusters map predominantly into a single
k=2 cluster, this shows k=2 conflates two access-differentiated but
both low-adoption groups -- pre